# Day 16 — Aggregates and NULL: COUNT, SUM, AVG, GROUP BY, HAVING, DISTINCT, COALESCE & CASE (9 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: turn rows into answers. Count, add up and average a table, split it into groups with GROUP BY, filter the groups with HAVING, remove duplicates with DISTINCT, handle missing values with COALESCE, and label rows with CASE. Underneath all of it: exactly what NULL does to every one of these.

- Deliverable: day16/report.sql, a content-ops report for Acme Retail: nine questions (entries per type, locale and status, owners sitting on drafts, missing prices, traffic buckets, stale drafts) answered against the given day16/setup.sql. Plus day16/dsa.py.

- Where this sits: Week 3, Phase 1 (Foundation), day 2 of 7. Yesterday you asked a table for rows; today you ask it for numbers. Day 17 joins several tables and groups across them, Day 20 adds window functions, and in Week 4 these queries become FastAPI dashboard endpoints.

Time split (5h): the Week 3 split, with the catch-up hour kept.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 6, typing and running every example in psql |
| SQL hands-on | 2h | report.sql: nine questions R1 to R9 |
| Catch-up | 1h | Day 15: finish schema.sql and queries.sql (Day 17 builds on them) |
| DSA | 0.5h | avg_ignoring_none and top_k_frequent in dsa.py |

Open items first: GitHub main still has one commit, from 28 Oct, and the Day 15 starters were sent as a patch because the push was blocked. Apply that patch first (git am on the file from yesterday's message) so day15/ exists on your laptop. Today's folder does not depend on Day 15: it ships its own tables in setup.sql. Push at the end of the day even if something is unfinished.

Before you start: Postgres running and the fde database from Day 15. Then, from inside day16/, run psql -d fde -v ON_ERROR_STOP=1 -f setup.sql once. It creates two tables: pages (8 rows) for every example below, and cms_entries (24 rows) for the task. Every output shown here was produced by running the query on PostgreSQL 16.

## Recap (2 minutes, from memory)

Three questions on Day 15. Answer out loud first; the answers are on the last page.

- You run INSERT INTO posts (uid, title) VALUES ("blt9", "Hi"); and get an error. What does the error say, and what is the one-character-type fix?

- Why can ORDER BY use a column alias made in SELECT, while WHERE cannot? Name the order in which the database works through the clauses.

- A "latest news" page shows the same article on page 1 and page 2. The query is ORDER BY published_at DESC LIMIT 10 OFFSET 10. What is the cause, and what do you add?

## Concept 1: Aggregate functions: many rows in, one value out

### Why it matters for an FDE

Most questions a customer asks are not "show me the rows" but "how many", "how much" and "what is typical": how many tickets are open, how much revenue sits in drafts, what is the average response time. Aggregates answer those inside the database, which sends back one small number instead of a million rows for your Python to loop over. When an LLM agent later writes SQL against a customer's warehouse, most of its queries will be aggregates, and you need to know when the number it returns is quietly wrong.

### How it works

An aggregate function takes a whole column of values and returns one value. The five you will use daily are count, sum, avg, min and max. Without GROUP BY (Concept 2) the whole table is one group, so the result is exactly one row. In JavaScript terms, an aggregate is a reduce() that the database runs for you.

The rule that catches everyone: aggregates skip NULL. count(*) counts rows. count(column) counts the rows where that column is not NULL. sum, avg, min and max all ignore NULLs, so avg(views) is the average over the pages that have views data, not over all pages.

```sql
-- Concept 1: count rows versus count values. pages has 8 rows.
SELECT count(*)          AS total_rows,   -- every row, NULLs or not
       count(author)     AS with_author,  -- rows where author IS NOT NULL
       count(word_count) AS counted,
       count(views)      AS with_views
FROM pages;
-- ->  total_rows | with_author | counted | with_views
-- -> ------------+-------------+---------+------------
-- ->           8 |           6 |       7 |          6
-- Like entries.filter(e => e.author != null).length in JS.
-- REMEMBER: count(*) counts rows; count(col) counts non-NULL values.
```

sum, avg, min and max work the same way. avg on integers returns an exact numeric with many decimals, so wrap it in round(value, places) for anything a person will read. min and max also work on text (alphabetical) and timestamps (earliest and latest).

```sql
-- Concept 1: the other four aggregates, all skipping NULL views.
SELECT sum(views)           AS total_views,
       round(avg(views), 1) AS avg_views,   -- avg over the 6 known
       min(word_count)      AS shortest,
       max(word_count)      AS longest
FROM pages;
-- ->  total_views | avg_views | shortest | longest
-- -> -------------+-----------+----------+---------
-- ->         7390 |    1231.7 |      150 |    1800
-- Without round(): 1231.6666666666666667 (integer avg gives numeric)

-- Treat missing views as 0 instead: a DIFFERENT question, a different answer
SELECT round(avg(coalesce(views, 0)), 1) AS avg_null_as_zero FROM pages;
-- ->  avg_null_as_zero
-- -> ------------------
-- ->             923.8      (7390 / 8 instead of 7390 / 6)
-- coalesce() is Concept 5; here it turns each NULL into 0 before avg.
```

Two behaviours to know before they surprise you in a report. On zero rows, count returns 0 but sum, avg, min and max return NULL, because the sum of nothing is unknown, not zero. And once a query has an aggregate, every other column in SELECT must be aggregated too, or the database cannot know which row's value to show.

```sql
-- Concept 1: zero rows in, and mixing a plain column with an aggregate.
-- psql meta-command: show NULL as (null), not blank. No ; and no
-- comment on the same line: psql would read the comment as arguments.
\pset null '(null)'
SELECT count(*) AS n, sum(views) AS s, max(views) AS m
FROM pages
WHERE locale = 'fr-fr';   -- there are no French pages
-- ->  n |   s    |   m
-- -> ---+--------+--------
-- ->  0 | (null) | (null)
-- GOTCHA: a total of NULL on a dashboard is "no rows", not "zero".

SELECT uid, count(*) FROM pages;
-- -> ERROR:  column "pages.uid" must appear in the GROUP BY clause or be
-- ->         used in an aggregate function
-- 8 uids, 1 count: which uid should the single row show? Concept 2.
```

### Real-world FDE example

In a discovery call, Acme's content lead says "almost all our products have prices". You run SELECT count(*), count(price) FROM cms_entries WHERE content_type = 'product'; and get 9 and 7. Two of nine products, 22 percent, would make the AI shopping assistant answer "I don't know the price". That one query turns a vague worry into a scoped data-cleanup task in your proposal.

### Common mistakes

- Reading avg(col) as the average over all rows. It is the average over the non-NULL rows. Decide which one the customer means, and use coalesce(col, 0) only when a missing value really means zero.

- Using count(col) when you meant rows. count(author) is not the number of pages; count(*) is.

- Showing an unrounded avg to a customer. Round it, and say in the report what was averaged.

- Expecting sum on an empty set to be 0. It is NULL; wrap it as coalesce(sum(x), 0) when you need a zero.

## Concept 2: GROUP BY: one result row per group

### Why it matters for an FDE

"Entries per locale", "tickets per priority", "revenue per region": every breakdown on every dashboard is a GROUP BY. It is the SQL version of the counting dict you wrote on Day 3 (counts[key] = counts.get(key, 0) + 1), except the database does it over millions of rows without sending them to you.

### How it works

GROUP BY column splits the rows that survived WHERE into buckets, one per distinct value, and then runs each aggregate once per bucket. The result has one row per group. It works like Object.groupBy() in modern JavaScript followed by a reduce on each group, or like the faceted counts in a CMS search sidebar ("Published (5) Draft (2)").

```sql
-- Concept 2: count pages per status, biggest group first.
SELECT status, count(*) AS pages
FROM pages
GROUP BY status              -- one bucket per distinct status
ORDER BY pages DESC, status; -- tie-breaker: status A to Z
-- ->   status   | pages
-- -> -----------+-------
-- ->  published |     5
-- ->  draft     |     2
-- ->  archived  |     1
-- Day 3 in Python: counts = {}; for p in pages:
--     counts[p["status"]] = counts.get(p["status"], 0) + 1
```

Group by several columns and you get one row per combination that exists in the data. Combinations with no rows do not appear at all: there is no "en-gb, draft, 0" row below, because no such page exists. If a report must show zeros, you need the pivot from Concept 6.

```sql
-- Concept 2: two grouping columns = one row per (locale, status) pair.
SELECT locale, status, count(*) AS pages, sum(views) AS views
FROM pages
GROUP BY locale, status
ORDER BY locale, status;
-- ->  locale |  status   | pages | views
-- -> --------+-----------+-------+--------
-- ->  en-gb  | published |     1 |     90
-- ->  en-us  | archived  |     1 |      0
-- ->  en-us  | draft     |     1 | (null)   <- its only page has NULL views
-- ->  en-us  | published |     3 |   7000
-- ->  hi-in  | draft     |     1 | (null)
-- ->  hi-in  | published |     1 |    300
```

The grouping rule: every column in SELECT must either appear in GROUP BY or sit inside an aggregate. The database is protecting you: inside the en-us group there are five different uids, and it refuses to pick one at random. One more surprise: NULL is treated as one group of its own, so all the unassigned pages land together in a row whose author is NULL.

```sql
-- Concept 2: the grouping rule, and NULL as its own group.
SELECT locale, uid, count(*) FROM pages GROUP BY locale;
-- -> ERROR:  column "pages.uid" must appear in the GROUP BY clause or be
-- ->         used in an aggregate function

SELECT author, count(*) AS pages
FROM pages
GROUP BY author
ORDER BY author;            -- NULLs sort LAST in ascending order
-- ->  author | pages
-- -> --------+-------
-- ->  Asha   |     2
-- ->  Meera  |     1
-- ->  Ravi   |     3
-- ->  (null) |     2      <- blt104 and blt107, both unassigned
-- FDE TIP: label that row for humans: coalesce(author, 'Unassigned').
```

### Real-world FDE example

Acme wants Hindi and Marathi sites at parity with English before Diwali. SELECT locale, count(*) FROM cms_entries GROUP BY locale; shows en-us 15, hi-in 5, mr-in 4. Grouped by locale and content type, it also shows which kinds of content are missing. That table becomes the translation backlog, and the same query run daily becomes the progress tile on the dashboard you build in Week 4.

### Common mistakes

- Adding a column to SELECT without adding it to GROUP BY, then "fixing" the error by grouping by an id. Grouping by uid makes every group one row, and the counts become all 1s.

- Forgetting that missing combinations do not appear. "No row" means zero, so say so in the report or build the pivot.

- Leaving out ORDER BY. Groups come back in no guaranteed order, the same as rows on Day 15.

- Grouping on messy text. 'Published' and 'published' are two groups. Check SELECT DISTINCT status (Concept 4) before you trust a breakdown.

## Concept 3: HAVING, and the full order of work

### Why it matters for an FDE

Many real questions filter on a total, not on a row: "authors with more than one draft", "locales with fewer than five live pages", "customers who opened ten tickets this week". WHERE cannot do this, because when WHERE runs the totals do not exist yet. Knowing which clause runs when is what lets you write these queries first time and explain an error in a live session.

### How it works

HAVING is a WHERE for groups: it runs after the groups are built and the aggregates are computed, and keeps only the groups whose condition is true. Extend the order of work from Day 15:

- FROM: take the table.

- WHERE: keep rows (no aggregates allowed here).

- GROUP BY: put the surviving rows into groups.

- HAVING: keep groups (aggregates allowed).

- SELECT: compute the output columns and their aliases.

- ORDER BY, then LIMIT: sort and cut the result.

```sql
-- Concept 3: WHERE filters rows first, HAVING filters groups after.
SELECT author, count(*) AS pages
FROM pages
WHERE status = 'published'      -- 1) only live pages go into groups
GROUP BY author                 -- 2) one group per author
HAVING count(*) >= 2            -- 3) keep authors with 2+ live pages
ORDER BY pages DESC, author;
-- ->  author | pages
-- -> --------+-------
-- ->  Asha   |     2
-- ->  Ravi   |     2      (Ravi's archived FAQ never reached a group)

SELECT author, count(*) FROM pages WHERE count(*) >= 2 GROUP BY author;
-- -> ERROR:  aggregate functions are not allowed in WHERE
-- WHERE runs before any group exists, so there is nothing to count yet.
```

Because HAVING runs before SELECT, it cannot use an alias you made in SELECT, the same reason WHERE cannot. Repeat the aggregate instead. Postgres is happy to compute count(*) once and use it in both places.

```sql
-- Concept 3: an alias cannot be used in HAVING.
SELECT locale, count(*) AS n
FROM pages
GROUP BY locale
HAVING n > 1;
-- -> ERROR:  column "n" does not exist
-- Write HAVING count(*) > 1 instead. ORDER BY n would have been fine.

SELECT locale, sum(views) AS views
FROM pages
GROUP BY locale
HAVING sum(views) > 500
ORDER BY views DESC;
-- ->  locale | views
-- -> --------+-------
-- ->  en-us  |  7000   (hi-in has 300, en-gb has 90)
-- REMEMBER: WHERE picks rows before grouping; HAVING picks groups after.
```

### Real-world FDE example

A support team asks which customers might churn: "anyone who opened more than five tickets in October". The date filter is about rows, so it goes in WHERE and shrinks the data before grouping. The "more than five" is about a total, so it goes in HAVING count(*) > 5. Putting the date filter in HAVING would still work, but it groups the whole year first and is slower and harder to read.

### Common mistakes

- An aggregate in WHERE. Move that condition to HAVING.

- A row condition in HAVING, like HAVING status <> 'archived' (it only works when status is grouped). Row conditions belong in WHERE, where they also make the query faster.

- Using the SELECT alias in HAVING. Repeat the expression.

## Concept 4: DISTINCT: unique values and unique counts

### Why it matters for an FDE

The first thing you do with an unfamiliar column is ask which values it really holds. SELECT DISTINCT status on a customer's ticket table often returns 'Open', 'open', 'OPEN ' and NULL, which is a data-quality finding before you have written any code. count(DISTINCT ...) answers questions like "how many unique entries" when one entry appears once per locale.

### How it works

SELECT DISTINCT removes duplicate result rows. It applies to the whole row you select, not to the first column. It is like [...new Set(values)] in JavaScript, or a Python set from Day 3. count(DISTINCT col) counts unique non-NULL values.

```sql
-- Concept 4: unique values, unique combinations, unique counts.
SELECT DISTINCT locale FROM pages ORDER BY locale;
-- ->  en-gb
-- ->  en-us
-- ->  hi-in

SELECT DISTINCT locale, status FROM pages ORDER BY locale, status;
-- -> 6 rows: every (locale, status) PAIR that exists, e.g.
-- ->  en-us | archived,  en-us | draft,  en-us | published, ...
-- DISTINCT applies to the whole selected row, not just to locale.

SELECT count(*)               AS rows,
       count(DISTINCT locale) AS locales,
       count(DISTINCT author) AS authors   -- NULL is not counted
FROM pages;
-- ->  rows | locales | authors
-- -> ------+---------+---------
-- ->     8 |       3 |       3
```

DISTINCT and GROUP BY overlap: SELECT DISTINCT locale and SELECT locale ... GROUP BY locale return the same rows. Use DISTINCT to list values, and GROUP BY as soon as you also want a number per value. Note that SELECT DISTINCT author does show NULL as a value, while count(DISTINCT author) does not count it.

```sql
-- Concept 4: count(DISTINCT) inside groups.
SELECT status, count(DISTINCT locale) AS locales
FROM pages
GROUP BY status
ORDER BY status;
-- ->   status   | locales
-- -> -----------+---------
-- ->  archived  |       1
-- ->  draft     |       2
-- ->  published |       3
-- GOTCHA: if a query returns duplicates you did not expect, find out why
-- before adding DISTINCT. On Day 17 a bad join creates duplicates, and
-- DISTINCT hides the bug instead of fixing it.
```

### Real-world FDE example

Acme's licence for the translation service is priced per unique entry, not per locale version. In cms_entries, the Home page appears three times (en-us, hi-in, mr-in) with the same uid. count(*) says 24, count(DISTINCT uid) says 16. Quoting 24 would have overstated the cost by half.

### Common mistakes

- Writing DISTINCT(locale), status and expecting only locale to be unique. DISTINCT is not a function; it always applies to the whole row.

- Sprinkling DISTINCT on a query to make duplicates disappear. Find the cause instead.

- Expecting count(DISTINCT col) to include NULL. It never does.

## Concept 5: NULL in depth: three-valued logic, COALESCE and NULLIF

### Why it matters for an FDE

Real customer data is full of holes: a CRM with no phone number, a ticket with no priority, a product with no price. NULL is how a database stores "unknown", and it follows rules that differ from JavaScript's null. Most "the numbers don't add up" incidents you will be called in for are NULL bugs, and they never raise an error.

### How it works

SQL logic has three values: true, false and unknown. Any comparison with NULL is unknown, any arithmetic with NULL is NULL, and WHERE keeps only rows where the condition is true. So author <> 'Ravi' silently drops the unassigned pages, because "is unknown different from Ravi?" is unknown. IS NULL, IS NOT NULL and IS DISTINCT FROM are the operators that treat NULL as a value you can test.

```sql
-- Concept 5: how NULL behaves in comparisons and arithmetic.
SELECT NULL = NULL                   AS eq,       -- unknown, not true
       NULL IS NULL                  AS is_null,
       1 + NULL                      AS plus,     -- NULL in, NULL out
       'a' || NULL                   AS joined,   -- || joins text
       5 IS DISTINCT FROM NULL       AS differs;  -- NULL-safe <>
-- ->    eq   | is_null |  plus  | joined | differs
-- -> --------+---------+--------+--------+---------
-- ->  (null) | t       | (null) | (null) | t
-- In JS, null === null is true. In SQL, NULL = NULL is unknown.

SELECT uid FROM pages WHERE author <> 'Ravi' ORDER BY uid;
-- ->  blt101, blt105, blt106     (3 rows: the unassigned are missing)
SELECT uid FROM pages WHERE author IS DISTINCT FROM 'Ravi' ORDER BY uid;
-- ->  blt101, blt104, blt105, blt106, blt107   (5 rows)
-- REMEMBER: any comparison with NULL is unknown, and WHERE drops unknown.
```

The worst NULL trap is NOT IN with a NULL in the list. locale NOT IN ('en-us', NULL) means locale <> 'en-us' AND locale <> NULL, and the second half is unknown for every row, so the query returns nothing. It happens in real life when the list comes from a subquery over a column that has NULLs.

```sql
-- Concept 5: NOT IN with a NULL in the list returns nothing.
SELECT uid FROM pages WHERE locale NOT IN ('en-us', NULL);
-- -> (0 rows)   no error, no warning, just an empty result
-- GOTCHA: keep NULLs out of NOT IN lists, or test IS NULL separately.
```

coalesce(a, b, ...) returns the first argument that is not NULL. It is SQL's version of JavaScript's a ?? b, and it is how you put a human label on a missing value or treat missing as zero. nullif(a, b) does the reverse: it returns NULL when a equals b. Its main use is division, where dividing by zero is an error but dividing by NULL just gives NULL.

```sql
-- Concept 5: coalesce (like ?? in JS) and nullif.
SELECT uid,
       coalesce(author, 'Unassigned') AS author,  -- label the unknown
       coalesce(views, 0)             AS views    -- only if NULL means 0
FROM pages
ORDER BY id
LIMIT 4;
-- ->   uid   |   author   | views
-- -> --------+------------+-------
-- ->  blt101 | Asha       |  5000
-- ->  blt102 | Ravi       |  1200
-- ->  blt103 | Ravi       |   300
-- ->  blt104 | Unassigned |     0

SELECT 10 / 0;
-- -> ERROR:  division by zero
SELECT 100 / nullif(0, 0) AS safe_div;
-- ->  safe_div
-- -> ----------
-- ->    (null)       nullif turned the 0 into NULL; 100 / NULL is NULL
-- FDE TIP: rates and percentages: x / nullif(total, 0), so one empty
-- group shows NULL instead of failing the whole dashboard query.
```

### Real-world FDE example

An insurer's report showed the average claim handling time getting better every month. The handling time was NULL for claims still open, and avg skipped them, so the slowest claims were left out until they closed. The honest report showed two numbers: the average for closed claims, and count(*) - count(closed_at) open claims. When you present a number built on a column with NULLs, say how the NULLs were treated.

### Common mistakes

- Assuming <> or NOT IN include the NULL rows. They never do. Add OR col IS NULL, or use IS DISTINCT FROM.

- Using coalesce(x, 0) everywhere. An unknown price is not a free product; replacing it with 0 makes averages and totals wrong in the other direction.

- Reading a blank cell in psql as an empty string. Run \pset null '(null)' so you can tell them apart. In Postgres, '' and NULL are different values.

## Concept 6: CASE: if/else inside a query

### Why it matters for an FDE

Customers think in categories that are not stored anywhere: "high-traffic pages", "stale drafts", "priority accounts". CASE lets you compute those labels inside the query and group by them. Combined with an aggregate, it also turns rows into columns, which is the "one row per locale with a column per status" shape every dashboard asks for.

### How it works

A searched CASE checks conditions top to bottom and returns the value of the first one that is true, like an if / else if / else chain or a JavaScript switch (true). If nothing matches and there is no ELSE, the result is NULL. A simple CASE compares one expression with several values, like a normal switch.

```sql
-- Concept 6: a searched CASE puts each page into a traffic bucket.
SELECT uid, views,
       CASE
           WHEN views IS NULL THEN 'no data'  -- test NULL FIRST
           WHEN views >= 1000 THEN 'high'     -- first true WHEN wins
           WHEN views >= 100  THEN 'medium'   -- so this means 100..999
           ELSE 'low'
       END AS traffic
FROM pages
ORDER BY id;
-- ->   uid   | views  | traffic
-- -> --------+--------+---------
-- ->  blt101 |   5000 | high
-- ->  blt102 |   1200 | high
-- ->  blt103 |    300 | medium
-- ->  blt104 | (null) | no data
-- ->  blt105 |    800 | medium
-- ->  blt106 | (null) | no data
-- ->  blt107 |     90 | low
-- ->  blt108 |      0 | low
```

Two traps follow from "first true condition wins". If you put the wide condition first (views >= 100 before views >= 1000), nothing ever reaches the second branch. And a NULL fails every comparison, so without its own branch it falls through to ELSE, here silently labelled 'low'. Group by the CASE alias to count per bucket:

```sql
-- Concept 6: group by a computed label; simple CASE; the NULL trap.
SELECT CASE WHEN views IS NULL THEN 'no data'
            WHEN views >= 1000 THEN 'high'
            WHEN views >= 100  THEN 'medium'
            ELSE 'low' END AS traffic,
       count(*) AS pages
FROM pages
GROUP BY traffic            -- Postgres lets GROUP BY use the alias
ORDER BY pages DESC, traffic;
-- -> high 2, low 2, medium 2, no data 2

-- simple CASE: compare one value; no ELSE means NULL for the rest
SELECT uid, CASE status WHEN 'published' THEN 'Live'
                        WHEN 'draft'     THEN 'In progress'
            END AS label
FROM pages WHERE uid IN ('blt101', 'blt104', 'blt108') ORDER BY uid;
-- ->  blt101 | Live
-- ->  blt104 | In progress
-- ->  blt108 | (null)        archived matched nothing, no ELSE
```

Conditional aggregation is CASE inside an aggregate. sum(CASE WHEN status = 'draft' THEN 1 ELSE 0 END) counts the drafts in each group. count(CASE WHEN ... THEN 1 END) does the same, because count skips the NULL that a missing ELSE produces. Postgres also has a shorter form, count(*) FILTER (WHERE ...). One row per locale with a column per status is called a pivot.

```sql
-- Concept 6: a pivot, three ways to count conditionally.
SELECT locale,
       count(*) AS total,
       count(*) FILTER (WHERE status = 'published') AS published,
       sum(CASE WHEN status = 'draft' THEN 1 ELSE 0 END) AS drafts,
       count(CASE WHEN status = 'archived' THEN 1 END) AS archived
FROM pages
GROUP BY locale
ORDER BY locale;
-- ->  locale | total | published | drafts | archived
-- -> --------+-------+-----------+--------+----------
-- ->  en-gb  |     1 |         1 |      0 |        0
-- ->  en-us  |     5 |         3 |      1 |        1
-- ->  hi-in  |     2 |         1 |      1 |        0
-- Zeros appear here (unlike GROUP BY locale, status), because every
-- locale row exists and the CASE just counts nothing.
-- REMEMBER: CASE picks the FIRST true branch; NULL needs its own WHEN.
```

### Real-world FDE example

Acme's head of content wants a Monday email: for each locale, how many pages are live, in draft and archived. With GROUP BY locale, status you would get nine rows and have to reshape them in Python. The pivot returns three rows that map one to one onto the email's table, and the zeros are already there. In Week 4 this exact query sits behind a FastAPI endpoint, and in Week 6 it becomes a tool an AI assistant can call.

### Common mistakes

- Ordering WHEN branches from narrow to wide wrongly. Put the most specific condition first, and test NULL first.

- Forgetting ELSE and getting NULLs in a label column. Add ELSE 'other' unless NULL is what you want.

- Using the CASE alias inside an expression in ORDER BY. ORDER BY traffic works, but ORDER BY CASE traffic WHEN ... fails with column does not exist. The task's R8 hint shows the fix.

- Writing sum(CASE WHEN ... THEN 1 END) and getting NULL for a group with no matches. Add ELSE 0, or use count or FILTER.

## Check your understanding (predict first, then run)

Write a prediction for each query in the notebook, then run it against pages. If a result surprises you, find which NULL or which clause caused it.

```sql
-- Check your understanding: predict each result, then run it.
\pset null '(null)'
-- 1. Three numbers. Which is not 8, and why?
SELECT count(*), count(views), count(DISTINCT status) FROM pages;
-- 2. Why are a and b different? (7 pages have a word count.)
SELECT round(avg(word_count)) AS a,
       sum(word_count) / count(*) AS b
FROM pages;
-- 3. How many rows, and what is in the last one?
SELECT author, count(*) FROM pages
WHERE status <> 'published'
GROUP BY author ORDER BY author;
-- 4. Which locales?
SELECT locale FROM pages GROUP BY locale HAVING max(views) > 1000;
-- 5. Four rows: in which order, and why is 'none' where it is?
SELECT coalesce(author, 'none') AS a, count(*) FROM pages
GROUP BY 1 ORDER BY 2 DESC, 1;   -- 1 and 2 mean "first/second column"
-- 6. Where does blt106 (word_count NULL) end up?
SELECT CASE WHEN word_count > 500 THEN 'long' ELSE 'short' END AS len,
       count(*)
FROM pages GROUP BY len ORDER BY len;
```

## Hands-on task (about 2 hours): day16/report.sql

The scenario: Acme Retail's content-ops lead runs a Monday review of their Contentstack stack. Until now an analyst built it by hand from CSV exports. You will answer her nine standing questions with SQL against cms_entries, the nightly sync table, so the report can be produced in seconds and later served by an API. The starter file has the brief, the exact expected output for each question, and hints above every TODO. Do not edit setup.sql; the expected outputs depend on its rows.

- Pull and load (10 min). Apply yesterday's patch if you have not, then git pull. From inside day16/, run psql -d fde -v ON_ERROR_STOP=1 -f setup.sql. Check with SELECT count(*) FROM cms_entries; (expect 24).

- Look before you count (10 min). Run \d cms_entries and SELECT DISTINCT content_type, locale, status FROM cms_entries ORDER BY 1, 2, 3;. Note which columns can be NULL. That tells you where count(*) and count(col) will differ.

- Answer R1 to R9 (90 min). Write one query per question under its TODO in report.sql. Run each alone in psql, compare it with its Expected comment, then save it.

- Run the whole report (5 min). psql -d fde -v ON_ERROR_STOP=1 -f report.sql. Every result must match.

- Commit and push. git add day16 && git commit -m "Day 16: content-ops report" && git push.

The nine questions (full wording and expected outputs are in the file):

| # | Question | Main idea |
|---|---|---|
| R1 | One-row summary: total, published, drafts, archived, unassigned | count(*), conditional counts, count(col) |
| R2 | Per content type: entries, unique entries, locales | GROUP BY, count(DISTINCT) |
| R3 | Per content type and status | GROUP BY two columns |
| R4 | Per locale: total plus one column per status | pivot with CASE or FILTER |
| R5 | Per owner: entries, published, views (Unassigned for NULL) | COALESCE, NULL group |
| R6 | Owners holding more than one draft | WHERE, GROUP BY, HAVING |
| R7 | Product price health: priced, missing, min, max, average | aggregates over NULLs |
| R8 | Traffic buckets in business order | CASE, sort key |
| R9 | Stale drafts (untouched 30+ days) per owner | dates, interval, min() |

Acceptance criteria:

- psql -d fde -v ON_ERROR_STOP=1 -f report.sql runs with no ERROR lines.

- Each of R1 to R9 matches its Expected comment exactly: the columns, their names, the row count and every value.

- Every multi-row result has an ORDER BY with a tie-breaker; no SELECT *.

- No NULL shows in an owner or label column (use coalesce), and no rounded value is shown unrounded.

- Each query has a one-line comment saying what the content team learns from it.

- The day16 folder is pushed to main.

Hints, if you are stuck (try for 10 minutes first):

- R1: one SELECT with five aggregates and no GROUP BY. "Unassigned" is the number of rows minus the number of rows that have an author.

- R2: one uid appears once per locale, so unique entries is a count of distinct uids.

- R4: start from Concept 6's pivot; the columns are total, published, draft, archived.

- R5: group by the raw author column, but show coalesce(author, 'Unassigned'). Views must show 0, not NULL, for an owner whose entries have no views.

- R6: which condition is about rows and which is about groups? One goes before GROUP BY, one after.

- R8: ORDER BY can use an alias only on its own. Add a second CASE that returns 1 to 4 as a sort_order column, group by both, and order by it.

- R9: "30 days before the report date" is timestamptz '2026-11-09 00:00+05:30' - interval '30 days'. Use that fixed date, not now(), so the expected output never changes. min(updated_at)::date shows the oldest update as a date.

Catch-up lane (1h): Day 15. Finish schema.sql and queries.sql from yesterday's patch. Done when psql -d fde -v ON_ERROR_STOP=1 -f schema.sql -f seed.sql runs twice with no ERROR and all ten queries match. Push it as its own commit before today's.

## DSA practice (30 min)

Two problems in day16/dsa.py that are today's SQL written by hand. Signatures, examples and hints are in the file; the expected outputs are in its __main__ block.

- avg_ignoring_none(values). Behave exactly like SQL's avg: skip None, and return None (not 0, not an error) when nothing is left. Hint: filter first, then check whether the filtered list is empty before you divide.

- top_k_frequent(words, k). Return the k most common words, most frequent first, ties broken alphabetically. This is LeetCode 692, and it is GROUP BY word ORDER BY count(*) DESC, word LIMIT k in Python. Hint: count with a dict (Day 3), then sorted() with a key that returns a tuple: a negative count, then the word.

## Quiz (3 quick questions)

- A customer's dashboard shows "average discount ₹350" from avg(discount), where orders with no discount have NULL. The CFO thinks it means the average across all orders. Is it? What two numbers would you show instead?

- Write the query: published entries per locale in cms_entries, only locales with at least 3, most first.

- Why does WHERE count(*) > 1 fail? Where does a status = 'draft' condition belong in a grouped query, and why there?

## What you achieved today

You learned how aggregates collapse rows into numbers and why they skip NULL, split data into groups with GROUP BY and filtered them with HAVING, found unique values with DISTINCT, handled missing data with COALESCE and NULLIF, and built labels and pivots with CASE. Your report.sql is the first piece of SQL in the repo that a real content team could run every Monday.

Tomorrow (Day 17): joins and modelling: primary and foreign keys, one-to-many and many-to-many, INNER and LEFT JOIN, and normalising entries, authors, tags and locales into linked tables.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 16 review": your recap answers, your six predictions next to the real outputs, report.sql (or the commit link), your dsa.py, your quiz answers and the Day 15 catch-up status. Note anything that felt unclear; tomorrow starts there.

## Recap answers

*Your answers:*
